# Servicio de mensajeria

Este es un ejemplo de ejecución. 

Si se quiere ejecutar por bloques, asegurarse de no cerrar la conexión a los clientes (último bloque).

In [12]:
from api import create_mongo_repository, create_neo4j_repository, create_redis_repository
from services import Application
import time 
import threading # para demostrar como blpop queda pendiente de un mensaje

### Inicializamos la aplicación
!!! Neo4j pide contraseña, en caso de no tener la que viene por defecto ("neo4j") se debe cambiar en la llamada a su función factory.
La contraseña es el segundo campo de la tupla.

In [ ]:
app = Application(
    mongo_repo=create_mongo_repository(),
    neo4j_repo=create_neo4j_repository(auth=("neo4j", "neo4j")),
    redis_repo=create_redis_repository()
)
# eliminamos todo lo que haya en las bases de datos
# esto no elimina snapshots
app.clear_databases()

# # descomentar esta linea para eliminar tambien los snapshots
# app.snapshotService.delete_all_snapshots()

pass

### Definimos los usuarios
También se hará una prueba con uno que tiene el número de teléfono repetido. Esto es para mostrar el funcionamiento del "constraint unique", ya que es importante que no se repitan teléfonos porque en redis se usarán como claves.

In [14]:
# CONTACTOS
contacts = [
    {"name": "Alice", "phone": "123456789"},
    {"name": "Boby", "phone": "987654321"},
    {"name": "Charlie", "phone": "555666777"},
    {"name": "Diana", "phone": "444555666"},
    {"name": "Eve", "phone": "222333444"},
    {"name": "Frank", "phone": "111222333"},
    {"name": "Grace", "phone": "777888999"},
]
duplicate_phone_contact = {"name": "Alice Duplicate", "phone": "123456789"}

for contact in contacts:
    app.contactService.create_user(name=contact["name"], phone=contact["phone"])

try:
    app.contactService.create_user(
        name=duplicate_phone_contact["name"],
        phone=duplicate_phone_contact["phone"]
    )  # Debería lanzar una excepción por violación de restricción única
except Exception as e:
    print(f"Error this phone number already exists: {e}")

Error this phone number already exists: {code: Neo.ClientError.Schema.ConstraintValidationFailed} {message: Node(59) already exists with label `User` and property `phone` = '123456789'}


### Creamos contactos entre los usuarios
Para simular el funcionamiento de una app real de mensajería, siempre buscaremos a los usuarios a través de su número, aunque aquí no sería necesario.

In [15]:
alice_id = app.contactService.search_one_user({"phone": "123456789"})["id"]
boby_id = app.contactService.search_one_user({"phone": "987654321"})["id"]
charlie_id = app.contactService.search_one_user({"phone": "555666777"})["id"]
diana_id = app.contactService.search_one_user({"phone": "444555666"})["id"]
eve_id = app.contactService.search_one_user({"phone": "222333444"})["id"]
frank_id = app.contactService.search_one_user({"phone": "111222333"})["id"]
grace_id = app.contactService.search_one_user({"phone": "777888999"})["id"]

# 2 maneras de crear contactos:
app.contactService._create_contact(
    from_user_props={"phone": "123456789"}, # Alice
    to_user_props={"phone": "987654321"} # Boby
)
app.contactService.create_contact_by_ids(
    from_user_id=alice_id,
    to_user_id=charlie_id
)
app.contactService.create_contact_by_ids(
    from_user_id=boby_id,
    to_user_id=diana_id
)
app.contactService.create_contact_by_ids(
    from_user_id=eve_id,
    to_user_id=diana_id
)
app.contactService.create_contact_by_ids(
    from_user_id=charlie_id,
    to_user_id=grace_id
)
app.contactService.create_contact_by_ids(
    from_user_id=frank_id,
    to_user_id=grace_id
)
app.contactService.create_contact_by_ids(
    from_user_id=frank_id,
    to_user_id=diana_id
)
app.contactService.create_contact_by_ids(
    from_user_id=boby_id,
    to_user_id=grace_id
)
app.contactService.create_contact_by_ids(
    from_user_id=alice_id,
    to_user_id=frank_id
)
pass


### Envío de mensajes
Primero se realiza una prueba con 2 usuarios que no están conectados.

Después, se activan las funciones de consumir mensaje. Estas se ejecutan en un thread distinto porque son bloqueantes y de esta manera simulan a otro usuario que tiene su aplicación esperando mensajes.

Además, dejaremos algunos mensajes sin consumir para que ver el funcionamiento de las snapshots con redis.

In [16]:
try:
    app.messageService.send_message(
        from_user_id=boby_id,
        to_user_id=charlie_id,
        message="Hello, Charlie!"
    )  # Debería lanzar un ValueError porque Boby y Charlie no están conectados
except ValueError as e:
    print(f"Error: {e}")


def consume_loop(app, from_user_id, to_user_id, label):
     while (msg := app.messageService.consume_message(
            from_user_id=from_user_id,
            to_user_id=to_user_id,
            timeout=5
        )) is not None:
        print(f"{label} recibe: {msg}")

t_boby = threading.Thread(
    target=consume_loop,
    args=(app, alice_id, boby_id, "Boby"),
    daemon=True
)

t_alice = threading.Thread(
    target=consume_loop,
    args=(app, boby_id, alice_id, "Alice"),
    daemon=True
)

t_boby.start()
t_alice.start()

time.sleep(1) 
app.messageService.send_message(
    from_user_id=alice_id,
    to_user_id=boby_id,
    message="Hola, Boby! Soy Alice."
)

time.sleep(1)
app.messageService.send_message(
    from_user_id=boby_id,
    to_user_id=alice_id,
    message="Hola, Alice!"
)
time.sleep(1)
app.messageService.send_message(
    from_user_id=boby_id,
    to_user_id=alice_id,
    message="Como estás?"
)
app.messageService.send_message(
    from_user_id=diana_id,
    to_user_id=eve_id,
    message="Hola, Eve!"
)

# guardamos una primera snapshot con pocos mensajes
app.snapshotService.create_snapshot()

# por ahora no consumimos más mensajes, así quedarán pendientes en Redis
app.messageService.send_message(
    from_user_id=alice_id,
    to_user_id=boby_id,
    message="Adios, Boby!"
)

app.messageService.send_message(
    from_user_id=frank_id,
    to_user_id=grace_id,
    message="Hola, Grace!"
)
app.messageService.send_message(
    from_user_id=grace_id,
    to_user_id=frank_id,
    message="Hola, Frank!"
)
app.messageService.send_message(
    from_user_id=grace_id,
    to_user_id=frank_id,
    message="como vas?"
)
app.messageService.send_message(
    from_user_id=frank_id,
    to_user_id=grace_id,
    message="Bien, gracias!"    
)
app.messageService.send_message(
    from_user_id=frank_id,
    to_user_id=grace_id,
    message="Y tu?"
)

app.messageService.send_message(
    from_user_id=frank_id,
    to_user_id=diana_id,
    message="Hola, Diana!"
)

app.messageService.send_message(
    from_user_id=alice_id,
    to_user_id=frank_id,
    message="Hola, Frank!"
)

pass


Error: Users are not connected.
Boby recibe: Hola, Boby! Soy Alice.
Alice recibe: Hola, Alice!
Alice recibe: Como estás?
Boby recibe: Adios, Boby!


### Snapshots
Realizamos una snapshot del estado actual de las 3 bases de datos y luego volvemos a comprobar que todo sigue funcionando tras restaurarla.

In [17]:
app.snapshotService.create_snapshot()

app.contactService.delete_users({"name": "Alice"})
print(f"Alice eliminada: {app.contactService.search_users({'name':'Alice'})}")

# crear otro snapshot para mostrar el listado
app.snapshotService.create_snapshot()
print(f"Lista de snapshots: {app.snapshotService.list_snapshots()}")


Alice eliminada: []
Lista de snapshots: [('version 1', '2026-01-11 20:34:11 UTC', {'total_users': 7, 'total_contacts': 9, 'total_messages': 4}), ('version 2', '2026-01-11 20:34:11 UTC', {'total_users': 7, 'total_contacts': 9, 'total_messages': 12}), ('version 3', '2026-01-11 20:34:11 UTC', {'total_users': 6, 'total_contacts': 6, 'total_messages': 7})]


Con este listado a través de la fecha y version podemos elegir que estado de las bases de datos restaurar.

In [18]:
app.snapshotService.restore_from_snapshot(1)
print(f"Alice vuelve a existir: {app.contactService.search_users({'name':'Alice'})}")

# los mensajes sin consumir se siguen podiendo consumir ahora
diana_id = app.contactService.search_one_user({"phone": "444555666"})["id"]
eve_id = app.contactService.search_one_user({"phone": "222333444"})["id"]
msg = app.messageService.consume_message(
    from_user_id=diana_id,
    to_user_id=eve_id,
    timeout=5
)
print(f"Eve recibe después del restore: {msg}")

Alice vuelve a existir: [{'phone': '123456789', 'name': 'Alice', 'id': '4:9e9cd09d-672f-4a30-988d-9d13b764e2fa:60', 'node_type': 'User'}]
Eve recibe después del restore: Hola, Eve!


Volvemos a probar todos los servicios de nuevo tras el restore.

In [19]:
app.contactService.create_user(name="Rosa", phone="999999999")
rosa_id = app.contactService.search_one_user({"phone": "999999999"})["id"]
charlie_id = app.contactService.search_one_user({"phone": "555666777"})["id"]
app.contactService.create_contact_by_ids(
    from_user_id=rosa_id,
    to_user_id=charlie_id
)

time.sleep(5)
app.messageService.send_message(
    from_user_id=rosa_id,
    to_user_id=charlie_id,
    message="Hola, Charlie! Soy Rosa."
)
app.messageService.send_message(
    from_user_id=rosa_id,
    to_user_id=charlie_id,
    message="Que tal?"
)

msg = ""
while msg is not None:
    msg = app.messageService.consume_message(
        from_user_id=rosa_id,
        to_user_id=charlie_id,
        timeout=5
    )
    if msg is not None:
        print(f"Charlie recibe: {msg}")

# dejamos este mensaje sin consumir para que haya algo en redis
app.messageService.send_message(
    from_user_id=charlie_id,
    to_user_id=rosa_id,
    message="Hola, todo bien, ¿y tu?"
)

app.snapshotService.create_snapshot()

Charlie recibe: Hola, Charlie! Soy Rosa.
Charlie recibe: Que tal?


### Consultas avanzadas
##### Neo4j

In [20]:
# vamos a usar el estado de la bdd de cuando habia mas mensajes
app.snapshotService.restore_from_snapshot(2)

alice_id = app.contactService.search_one_user({"phone": "123456789"})["id"]
diana_id = app.contactService.search_one_user({"phone": "444555666"})["id"]

print("Contactos de alice: ", app.contactService.get_contacts_of_user(alice_id))
print("Contactos de alice ordenados por mensajes enviados: ", app.contactService.get_contacts_ordered_by_messages(alice_id))

camino = app.contactService.shortest_path_max_messages(alice_id, diana_id)
print("Camino entre Alice y Diana con más mensajes: ", camino)

Contactos de alice:  [{'phone': '111222333', 'name': 'Frank', 'id': '4:9e9cd09d-672f-4a30-988d-9d13b764e2fa:18'}, {'phone': '555666777', 'name': 'Charlie', 'id': '4:9e9cd09d-672f-4a30-988d-9d13b764e2fa:8'}, {'phone': '987654321', 'name': 'Boby', 'id': '4:9e9cd09d-672f-4a30-988d-9d13b764e2fa:14'}]
Contactos de alice ordenados por mensajes enviados:  [{'phone': '987654321', 'name': 'Boby', 'id': '4:9e9cd09d-672f-4a30-988d-9d13b764e2fa:14', 'message_count': 4}, {'phone': '111222333', 'name': 'Frank', 'id': '4:9e9cd09d-672f-4a30-988d-9d13b764e2fa:18', 'message_count': 1}, {'phone': '555666777', 'name': 'Charlie', 'id': '4:9e9cd09d-672f-4a30-988d-9d13b764e2fa:8', 'message_count': 0}]
Camino entre Alice y Diana con más mensajes:  [{'phone': '123456789', 'name': 'Alice', 'id': '4:9e9cd09d-672f-4a30-988d-9d13b764e2fa:56'}, {'phone': '987654321', 'name': 'Boby', 'id': '4:9e9cd09d-672f-4a30-988d-9d13b764e2fa:14'}, {'phone': '777888999', 'name': 'Grace', 'id': '4:9e9cd09d-672f-4a30-988d-9d13b764e

##### Mongo

In [21]:
# Devuelven array, asi que si solo se quiere el conteo se puede usar len()

# Historial de mensajes entre Alice y Boby por orden de tiempo
boby_id = app.contactService.search_one_user({"phone": "987654321"})["id"]
conver = app.messageService.message_history_by_connection(
    from_user_id=boby_id,
    to_user_id=alice_id
)
print("Historial de mensajes entre Alice y Boby: ", conver)
print("Número de mensajes entre Alice y Boby: ", len(conver))

# Historial de mensajes de Alice
msgs_user = app.messageService.message_history_by_user(
    user_id=alice_id
)
print("Historial de mensajes de Alice: ", msgs_user)
print("Número de mensajes de Alice: ", len(msgs_user))

Historial de mensajes entre Alice y Boby:  [{'timestamp': datetime.datetime(2026, 1, 11, 20, 34, 9, 392000), 'from_phone': '123456789', 'to_phone': '987654321', 'message': 'Hola, Boby! Soy Alice.'}, {'timestamp': datetime.datetime(2026, 1, 11, 20, 34, 10, 414000), 'from_phone': '987654321', 'to_phone': '123456789', 'message': 'Hola, Alice!'}, {'timestamp': datetime.datetime(2026, 1, 11, 20, 34, 11, 436000), 'from_phone': '987654321', 'to_phone': '123456789', 'message': 'Como estás?'}, {'timestamp': datetime.datetime(2026, 1, 11, 20, 34, 11, 484000), 'from_phone': '123456789', 'to_phone': '987654321', 'message': 'Adios, Boby!'}]
Número de mensajes entre Alice y Boby:  4
Historial de mensajes de Alice:  [{'timestamp': datetime.datetime(2026, 1, 11, 20, 34, 9, 392000), 'from_phone': '123456789', 'to_phone': '987654321', 'message': 'Hola, Boby! Soy Alice.'}, {'timestamp': datetime.datetime(2026, 1, 11, 20, 34, 10, 414000), 'from_phone': '987654321', 'to_phone': '123456789', 'message': 'Hol

### Cierre del programa
El método app.close() se encarga de cerrar los clientes de neo4j, redis y mongo.

El sleep es para esperar a que los blpop dejen de estar escuchando, porque si no se espera da error.

In [22]:
time.sleep(10)  # esperamos a que los hilos terminen de ejecutar
app.close()
